# Milestone 1: Automated Pipeline for arXiv Metadata, TeX Sources, and References

Notebook này giữ cách trình bày theo **notebook thu thập dữ liệu ở nhánh main**, với các bước rõ ràng và **7 cell code chính**. Nhóm lưu mã nguồn, notebook và Jupyter Book trong **một repository**; dữ liệu crawl nằm trên Google Drive.

### Tổng quan quy trình Milestone 1

1. **Khám phá metadata (Bảng 1):** đọc papers.jsonl, kiểm tra số bài báo và các version dự kiến. Crawler metadata cũ vẫn được giữ nhưng **không tự chạy lại** trong notebook rút gọn.
2. **Tải source LaTeX (Bảng 2):** tải từng version v1/v2/... từ arXiv, lưu ở data/raw/<id>/vN/ và xác minh TeX/Bib bằng marker + SHA-256. Bảng m1_download_versions.csv mới phản ánh kết quả từng version.
3. **Thu thập references (Bảng 3):** lưu cả các raw references không có arXiv ID từ Semantic Scholar. Bảng references.jsonl và m1_reference_stats.csv cũ không bị thay đổi.
4. **Benchmark hệ thống (Bảng 4):** giữ bảng legacy để đối chiếu. Chưa xem đây là benchmark CPU-only chính thức của pipeline theo version; phải đo riêng trên Colab khi nghiệm thu.

### Cơ chế chống lỗi và phục hồi

- **Dry-run mặc định:** không tải source, không gọi Semantic Scholar và không xuất CSV/JSON mới nếu chưa chủ động bật cờ.
- **Retry, backoff và resume:** bỏ qua source đã được xác minh; ghi nhận lỗi HTTP 404/429/5xx và thử lại theo chính sách của từng collector.
- **An toàn dữ liệu:** clone/pull mã nguồn ở /content, không làm Git trực tiếp trên Google Drive; không ghi đè thư mục legacy.
- **Kiểm định:** sau lượt crawl, so khớp metadata, hash file và ledger; kết quả được tái sử dụng khi tổng kết để tránh quét Drive nhiều lần.

**Cách chạy:** chạy 7 cell code theo thứ tự. Với lần test đầu, giữ mọi cờ DOWNLOAD/FETCH/EXPORT ở False. Sau khi tải thêm dữ liệu, chạy lại cell 6 rồi cell 7.


## 1. Kết nối Google Drive và Thiết lập Môi trường (trên Colab)

Cell code **1/7** thực hiện: (1) Mount Google Drive; (2) clone/pull mã nguồn của nhánh đang phát triển vào /content/arxiv-code; (3) đặt đường dẫn dữ liệu và cài thư viện từ requirements.txt.

**Vì sao code và data ở hai nơi?** Git hoạt động trên ổ cục bộ Colab để tránh lỗi .git/index.lock trên Drive. Dữ liệu đã tải trước đây vẫn được giữ nguyên ở thư mục data trên Drive. Khi mở runtime mới, chạy lại cell này; nếu báo import nhầm config, chọn **Runtime → Restart session**.

Sau khi in **Môi trường đã sẵn sàng**, tiếp tục cấu hình, không chạy thêm cell clone trên Drive như phiên bản cũ.


In [ ]:
# 1/7 — KẾT NỐI DRIVE VÀ THIẾT LẬP MÔI TRƯỜNG
# Code được clone trên /content; data vẫn nằm ở Google Drive.
import os, sys, subprocess
from pathlib import Path

BRANCH = "feat/milestone1-compliance"
try:
    from google.colab import drive
except ImportError:
    CODE_DIR = Path.cwd().resolve()
    DATA_DIR = Path(os.environ.get("ARXIV_DATA_DIR", CODE_DIR / "data")).resolve()
else:
    drive.mount("/content/drive")
    CODE_DIR = Path("/content/arxiv-code")
    DATA_DIR = Path(os.environ.get(
        "ARXIV_DATA_DIR",
        "/content/drive/MyDrive/Colab_Notebooks/arXiv-Project/data",
    )).resolve()
    if not (CODE_DIR / ".git").is_dir():
        subprocess.run([
            "git", "clone", "--depth", "1", "--single-branch",
            "--branch", BRANCH,
            "https://github.com/ahtuan171/arXiv-Project.git", str(CODE_DIR),
        ], check=True)
    else:
        actual = subprocess.check_output(
            ["git", "-C", str(CODE_DIR), "branch", "--show-current"], text=True
        ).strip()
        if actual != BRANCH:
            raise RuntimeError("Wrong branch; restart runtime and re-open the GitHub notebook")
        subprocess.run(
            ["git", "-C", str(CODE_DIR), "pull", "--ff-only", "origin", BRANCH],
            check=True,
        )

if not (DATA_DIR / "papers.jsonl").is_file():
    raise FileNotFoundError(f"Missing original metadata: {DATA_DIR / 'papers.jsonl'}")
os.environ["ARXIV_DATA_DIR"] = str(DATA_DIR)
os.chdir(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
if "src.config" in sys.modules:
    from src import config
    if Path(config.DEFAULT_DATA_DIR).resolve() != DATA_DIR:
        raise RuntimeError("Cấu hình cũ đang được sử dụng; hãy Restart session.")
else:
    from src import config
assert Path(config.DEFAULT_DATA_DIR).resolve() == DATA_DIR
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_DIR / "requirements.txt")],
    check=True,
)
print("✅ Môi trường đã sẵn sàng")
print("Mã nguồn:", CODE_DIR, "| Dữ liệu:", DATA_DIR)
subprocess.run(["git", "-C", str(CODE_DIR), "log", "-1", "--oneline"], check=True)


### 1.1. Cấu hình Semantic Scholar API Key (Tùy chọn)

Tương tự notebook gốc, bạn có thể thêm key trong **Colab Secrets**: chọn biểu tượng chìa khóa ở thanh bên trái, tạo secret tên **S2_API_KEY**, dán API key rồi bật **Notebook access**.

Key sẽ được nạp tự động trong **cell 2/7**. Không gửi một request thử key khi cấu hình, để tránh tốn quota hoặc gây lỗi 429. Không có key vẫn có thể chạy ở quota ẩn danh nhưng cần thận trọng khi thu thập nhiều paper.


## 2. Khai báo khoảng arXiv-ID cần xử lý

Cell code **2/7** là biểu mẫu cấu hình kiểu **Colab form** như notebook trên main.

- **start_id, end_id, LIMIT:** chọn dải paper. LIMIT=0 nghĩa là toàn bộ dải được chọn, không đồng nghĩa tải vô hạn.
- **DOWNLOAD_VERSIONS, VERSION_LIMIT:** bật tải thật và giới hạn **số lượt thử download**; đã có source hợp lệ thì resume bỏ qua.
- **FETCH_RAW_REFERENCES, RAW_REF_LIMIT:** bật thu thập và giới hạn **số paper truy vấn Semantic Scholar**.
- **Retry:** chỉ bật khi cần kiểm tra lại lỗi/404; mặc định False.
- **EXPORT...:** chỉ ghi các bảng thống kê khi bật từng cờ; mặc định False.

Với lần chạy thử đầu tiên, giữ mọi cờ thu thập/xuất file ở False, sau đó tăng dần giới hạn khi kết quả đã ổn.


In [ ]:
#@title Cấu hình tham số Pipeline (Khoảng ID, giới hạn, Retry và xuất báo cáo) { display-mode: "form" }
# 2/7 — KHAI BÁO CẤU HÌNH MILESTONE 1
#@markdown ### 1. Khoảng arXiv ID và giới hạn
#@markdown Nhập start_id, end_id theo định dạng YYMM.NNNNN. LIMIT=0 dùng toàn bộ khoảng ID đã chọn.

start_id = "2502.13235"  #@param {type:"string"}
end_id = "2502.13245"  #@param {type:"string"}
LIMIT = 0                         # 0 = all IDs in selected range  #@param {type:"integer"}

#@markdown ### 2. Kiểm thử an toàn
#@markdown Khi bật, chạy pytest offline trên dữ liệu giả, không gọi arXiv/Semantic Scholar.
RUN_OFFLINE_TESTS = True  #@param {type:"boolean"}
#@markdown ### 3. Tải LaTeX source theo từng version
#@markdown Mặc định không tải. VERSION_LIMIT giới hạn số lượt THỬ qua mạng, không phải số paper.
DOWNLOAD_VERSIONS = False         # Default: no source requests/writes  #@param {type:"boolean"}
VERSION_LIMIT = 3                 # Max NETWORK ATTEMPTS per run  #@param {type:"integer"}
VERSION_RETRY_ERRORS = False  #@param {type:"boolean"}
VERSION_RETRY_NOT_FOUND = False  #@param {type:"boolean"}

#@markdown ### 4. Thu thập raw references
#@markdown Mặc định không gọi API. RAW_REF_LIMIT là số paper tối đa gửi request.
FETCH_RAW_REFERENCES = False      # Default: no S2 requests/writes  #@param {type:"boolean"}
RAW_REF_LIMIT = 3                 # Max papers queried per run  #@param {type:"integer"}
RAW_REF_RETRY_ERRORS = False  #@param {type:"boolean"}
RAW_REF_RETRY_NOT_FOUND = False  #@param {type:"boolean"}

#@markdown ### 5. Xuất các file thống kê mới
#@markdown Giữ False khi test. Mỗi cờ chỉ ghi một file báo cáo riêng, không sửa dữ liệu legacy.
EXPORT_VERSION_CSV = False  #@param {type:"boolean"}
EXPORT_VERSION_METADATA = False  #@param {type:"boolean"}
EXPORT_M1_INTEGRITY_REPORT = False  #@param {type:"boolean"}
EXPORT_M1_REPORT = False  #@param {type:"boolean"}
#@markdown ### 6. Giới hạn số ví dụ lỗi hiển thị
AUDIT_SAMPLE_SIZE = 10  #@param {type:"integer"}

from src import config
from src.utils import parse_arxiv_range
id_list = parse_arxiv_range(start_id, end_id)
if LIMIT > 0:
    id_list = id_list[:LIMIT]
if not id_list:
    raise ValueError("Khoảng ID rỗng, hãy kiểm tra start_id và end_id")
if VERSION_LIMIT < 1 or RAW_REF_LIMIT < 1:
    raise ValueError("VERSION_LIMIT và RAW_REF_LIMIT phải >= 1 để thử tải an toàn")

try:
    from google.colab import userdata
    secret = userdata.get("S2_API_KEY") or ""
except Exception:
    secret = os.environ.get("S2_API_KEY", "")
if secret.strip():
    config.S2_API_KEY = secret.strip()
    os.environ["S2_API_KEY"] = secret.strip()
print(f"✅ Đã cấu hình: {len(id_list)} IDs ({id_list[0]} to {id_list[-1]})")
print("Semantic Scholar API:", "API key ready" if config.S2_API_KEY else "anonymous")
print("Tải source:", "LIVE" if DOWNLOAD_VERSIONS else "CHỈ XEM",
      "| Raw references:", "LIVE" if FETCH_RAW_REFERENCES else "DRY-RUN")
VALIDATION_REQUIRED = True


## 3. Kiểm tra dữ liệu ban đầu và Môi trường (Preflight)

Cell **3/7** đọc nhanh metadata và manifest để thống kê số paper, số version và ứng viên cần xử lý. Nó **không đọc/hash toàn bộ các thư mục source** ở bước này.

Nếu RUN_OFFLINE_TESTS=True, notebook chạy bộ pytest trên dữ liệu giả; các test bao gồm downloader, resume/recovery, raw references và integrity. Đây là kiểm thử code, **không chứng minh dữ liệu trên Drive đã đầy đủ**.

**Bảng 4 — Benchmark:** notebook main có phần khởi động tracker trước khi crawl. Trong pipeline mới, chưa đo lại toàn bộ luồng vN bằng CPU-only; không sử dụng benchmark cũ để tuyên bố kết quả mới.


In [ ]:
# 3/7 — KIỂM TRA METADATA VÀ TEST OFFLINE
# Không gửi request crawl hoặc ghi dữ liệu Drive.
from src.scripts.audit_missing_versions import audit_missing_versions

preflight = audit_missing_versions(
    config.DEFAULT_DATA_DIR, ids=id_list, sample_size=AUDIT_SAMPLE_SIZE
)
print("=== KIỂM TRA METADATA (CHỈ ĐỌC) ===")
for key, val in preflight["summary"].items():
    print(f"{key:31}: {val}")
for warning in preflight.get("warnings", [])[:AUDIT_SAMPLE_SIZE]:
    print("Cảnh báo metadata:", warning)

if RUN_OFFLINE_TESTS:
    OFFLINE_TESTS = [
        "tests/test_version_downloader.py",
        "tests/test_audit_version_downloads.py",
        "tests/test_audit_legacy_matches.py",
        "tests/test_audit_progress.py",
        "tests/test_build_version_metadata.py",
        "tests/test_raw_reference_collector.py",
        "tests/test_audit_m1_integrity.py",
        "tests/test_m1_offline_end_to_end.py",
        "tests/test_build_m1_report.py",
        "tests/test_notebook_layout.py",
    ]
    assert all((CODE_DIR / path).is_file() for path in OFFLINE_TESTS)
    child_env = dict(os.environ, ARXIV_DATA_DIR="/tmp/m1-fixtures-only",
                     S2_API_KEY="", PYTHONDONTWRITEBYTECODE="1",
                     PYTEST_DISABLE_PLUGIN_AUTOLOAD="1")
    subprocess.run(
        [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", *OFFLINE_TESTS],
        cwd=str(CODE_DIR), env=child_env, check=True,
    )
    print("✅ Bộ kiểm thử offline đã đạt.")
else:
    print("Đã bỏ qua kiểm thử offline.")


## 4. Thực thi Pipeline Thu thập Dữ liệu

Theo bố cục **Thực thi Toàn bộ Pipeline** của notebook main, quá trình mới chia thành hai công đoạn có thể chạy độc lập, có giới hạn và resume. Cả hai đều **dry-run mặc định** để tránh gửi request ngoài ý muốn.

### 4.1. Tải mã nguồn LaTeX theo từng version (Bảng 2)

Cell **4/7** đọc danh sách version cần tải, kiểm tra marker/hash của các version đã có và đưa ra kế hoạch. Khi DOWNLOAD_VERSIONS=True, nó tải tối đa VERSION_LIMIT lượt thử, lưu vào raw/<paper>/vN/ và cập nhật versions_manifest.jsonl.

Thanh tiến độ phản ánh **số lượt thử tải đã xử lý**, không phải tỷ lệ tải thành công. Các trạng thái như not_found, failed, conflict cần được kiểm tra trước khi tăng giới hạn. Thư mục version có dữ liệu không đáng tin sẽ không bị ghi đè tự động.


In [ ]:
# 4/7 — TẢI SOURCE THEO PHIÊN BẢN
# Chỉ tải thật khi DOWNLOAD_VERSIONS=True. Có giới hạn lượt thử.
from tqdm.auto import tqdm
from src.download_module.version_downloader import run_version_downloads

if DOWNLOAD_VERSIONS:
    VALIDATION_REQUIRED = True
with tqdm(desc="Tải source vN", unit="attempt", dynamic_ncols=True,
          disable=not DOWNLOAD_VERSIONS) as bar:
    def on_source_progress(done, total, event):
        if bar.total != total:
            bar.total = total
            bar.refresh()
        bar.update(done - bar.n)
        bar.set_postfix_str(event["status"])
    version_result = run_version_downloads(
        config.DEFAULT_DATA_DIR, ids=id_list,
        max_downloads=VERSION_LIMIT, dry_run=not DOWNLOAD_VERSIONS,
        retry_errors=VERSION_RETRY_ERRORS,
        retry_not_found=VERSION_RETRY_NOT_FOUND,
        progress_callback=on_source_progress if DOWNLOAD_VERSIONS else None,
    )
print("=== BẢNG 2 | SOURCE THEO VERSION ===")
print("Kế hoạch:", version_result["plan"]["summary"])
print("Kết quả:", version_result["summary"])
if not DOWNLOAD_VERSIONS:
    print("✅ Chỉ xem kế hoạch, chưa tải source.")


### 4.2. Thu thập raw references từ Semantic Scholar (Bảng 3)

Cell **5/7** giữ các trích dẫn ngay cả khi không có arXiv ID hoặc thiếu thông tin citedPaper. Khi bật FETCH_RAW_REFERENCES=True, collector thực hiện phân trang, chỉ công nhận paper thành công sau khi đã lấy đủ các trang API.

Dữ liệu được lưu ở references_raw.jsonl và references_raw_ledger.jsonl. Các reference này là dữ liệu **cấp paper**; không tự suy diễn rằng chúng thuộc v1 hoặc v2. Bảng references.jsonl cũ vẫn được giữ nguyên.

RAW_REF_LIMIT là số paper cần gọi mạng trong lượt chạy; progress bar hiển thị tiến độ theo paper.


In [ ]:
# 5/7 — CÀO RAW REFERENCES
# Giữ cả references không có arXiv ID; không tự gán cho version vN.
from tqdm.auto import tqdm
from src.crawl_references_module.raw_reference_collector import collect_raw_references

if FETCH_RAW_REFERENCES:
    VALIDATION_REQUIRED = True
with tqdm(desc="Cào raw references", unit="paper", dynamic_ncols=True,
          disable=not FETCH_RAW_REFERENCES) as bar:
    def on_raw_progress(done, total, event):
        if bar.total != total:
            bar.total = total
            bar.refresh()
        bar.update(done - bar.n)
        bar.set_postfix_str(event["status"])
    raw_ref_result = collect_raw_references(
        config.DEFAULT_DATA_DIR, ids=id_list,
        max_papers=RAW_REF_LIMIT, dry_run=not FETCH_RAW_REFERENCES,
        retry_errors=RAW_REF_RETRY_ERRORS,
        retry_not_found=RAW_REF_RETRY_NOT_FOUND,
        progress_callback=on_raw_progress if FETCH_RAW_REFERENCES else None,
    )
print("=== BẢNG 3 | RAW REFERENCES ===")
print(raw_ref_result["summary"])
if not FETCH_RAW_REFERENCES:
    print("✅ Chưa gọi Semantic Scholar hoặc ghi dữ liệu.")


### Chạy lại và phục hồi các ID bị lỗi

1. **Colab bị ngắt:** khởi động runtime mới, chạy lại cell 1 và 2, sau đó tiếp tục công đoạn đang làm. Không xóa dữ liệu cũ.
2. **HTTP 429, 5xx, lỗi mạng:** xem thời gian cooldown/next_retry_at trong manifest hoặc ledger; chạy lại khi đã khắc phục nguyên nhân.
3. **HTTP 404:** mặc định bỏ qua, chỉ bật *_RETRY_NOT_FOUND khi cần thử lại thật sự.
4. **HTTP 403 hoặc conflict:** kiểm tra quyền, marker và thư mục version trước khi bật retry; không ghi đè/xóa thủ công khi chưa biết nguyên nhân.
5. **Sau khi dữ liệu thay đổi:** chạy lại cell 6 (validation), rồi mới chạy cell 7 (summary); không dùng kết quả audit cũ.

Các file versions_manifest.jsonl và references_raw_ledger.jsonl lưu lịch sử retry/resume. Kết quả thành công chỉ được xác nhận khi nội dung và hash còn hợp lệ.


## 5. Kiểm tra và Trích xuất Toàn bộ 4 Bảng Thống kê Milestone 1

Giữ cách hiển thị Bảng 1–4 của notebook trên main, nhưng **phân biệt bảng legacy với bảng version-aware**:

| Bảng | File | Ý nghĩa |
|---|---|---|
| 1. Discovery | m1_discovery_stats.csv | Metadata cấp paper đã thu thập trước đây |
| 2. Download | m1_download_stats.csv (cũ) và m1_download_versions.csv (mới) | Source cấp paper so với source từng version |
| 3. References | m1_reference_stats.csv (cũ) và references_raw.jsonl | Trích dẫn có arXiv ID so với raw references |
| 4. Benchmark | m1_system_benchmark.csv (cũ) | Chỉ tham khảo; chưa phải benchmark mới |

### 5.1. Kiểm định phiên bản và dữ liệu đầu ra

Cell **6/7** xác minh source bằng SHA-256, kiểm tra metadata index, raw-reference ledger và các snapshot đã xuất. Quy trình **quét hash một lượt**, rồi tái sử dụng kết quả đó cho integrity audit.

- **needs_fix:** xuất hiện dữ liệu mâu thuẫn/hỏng.
- **in_progress:** chưa tải hoặc chưa kiểm tra/xuất đủ; bình thường khi mới test.
- **structurally_consistent:** các điều kiện cấu trúc đã đạt; không phải giấy chứng nhận hoàn tất Milestone 1.

Khi bật EXPORT_VERSION_CSV, EXPORT_VERSION_METADATA hoặc EXPORT_M1_INTEGRITY_REPORT ở cell 2, cell này mới ghi các file thống kê mới tương ứng. Trên Drive, kiểm tra nhiều nghìn version bằng hash có thể mất thời gian.


In [ ]:
# 6/7 — KIỂM ĐỊNH SOURCE, METADATA VÀ REFERENCES
# Quét SHA-256 một lượt, tái sử dụng dữ liệu đã kiểm tra.
from tqdm.auto import tqdm
from src.scripts.audit_version_downloads import audit_version_downloads, export_version_csv
from src.scripts.build_version_metadata import build_version_metadata, export_version_metadata
from src.scripts.audit_m1_integrity import audit_m1_integrity, export_integrity_report

with tqdm(desc="Xác minh hash của version", unit="version", dynamic_ncols=True) as bar:
    def on_verify(done, total, record):
        if bar.total != total:
            bar.total = total
            bar.refresh()
        bar.update(done - bar.n)
        bar.set_postfix_str(record["status"])
    version_audit = audit_version_downloads(
        config.DEFAULT_DATA_DIR, sample_size=AUDIT_SAMPLE_SIZE,
        progress_callback=on_verify,
    )
metadata_report = build_version_metadata(
    config.DEFAULT_DATA_DIR, sample_size=AUDIT_SAMPLE_SIZE,
)
# Update requested snapshots BEFORE validating snapshot freshness.
if EXPORT_VERSION_CSV:
    print("Bảng version CSV:", export_version_csv(version_audit))
if EXPORT_VERSION_METADATA:
    print("Metadata theo version:", export_version_metadata(metadata_report))

integrity_report = audit_m1_integrity(
    config.DEFAULT_DATA_DIR, sample_size=AUDIT_SAMPLE_SIZE,
    version_report=version_audit, metadata_report=metadata_report,
)
VALIDATION_REQUIRED = False
print("=== TRẠNG THÁI KIỂM ĐỊNH:", integrity_report["status"], "===")
print("Bao phủ:", integrity_report["coverage"])
print("Source:", integrity_report["source_versions"])
print("Raw references:", integrity_report["raw_references"])
for label in ("blockers", "incomplete", "warnings"):
    for item in integrity_report[label][:AUDIT_SAMPLE_SIZE]:
        print(label.upper(), ":", item)
if EXPORT_M1_INTEGRITY_REPORT:
    print("Báo cáo kiểm định:", export_integrity_report(integrity_report))
else:
    print("✅ Đã kiểm định, chưa xuất báo cáo ra Drive.")


### 5.2. Tổng hợp kết quả và trình bày Bảng 1–4

Cell **7/7** hiển thị tỷ lệ version có source đã xác minh, trạng thái references, thống kê kích thước (nếu có snapshot hợp lệ), các vấn đề cần sửa và **bốn bảng CSV theo phong cách notebook main**.

Bảng 2 mới theo version được hiển thị riêng; các bảng legacy được gắn nhãn rõ để không bị nhầm với kết quả pipeline mới. Bật EXPORT_M1_REPORT=True mới ghi data/reports/m1_summary.json phục vụ Jupyter Book trong **cùng repository**.

**Lưu ý:** những chỉ số chưa đo thực tế trên Colab CPU-only (wall time, RAM, disk) vẫn được ghi là thiếu bằng chứng, tuyệt đối không dùng bảng benchmark cũ làm số liệu mới.


In [ ]:
# 7/7 — TỔNG HỢP CÁC BẢNG THỐNG KÊ VÀ BÁO CÁO
# Dùng kết quả đã xác minh từ cell 6, không hash source lại.
from src.scripts.build_m1_report import build_m1_report, export_m1_report
from IPython.display import display
import pandas as pd

if "integrity_report" not in globals() or VALIDATION_REQUIRED:
    raise RuntimeError("Sau khi dữ liệu thay đổi, phải chạy lại cell 6.")
m1_summary = build_m1_report(
    config.DEFAULT_DATA_DIR, integrity_report=integrity_report,
)
c = m1_summary["coverage"]
display(pd.DataFrame([
    {"Metric": "Valid metadata papers", "Value": c["valid_metadata_papers"]},
    {"Metric": "Expected source versions", "Value": c["expected_versions"]},
    {"Metric": "Verified source versions", "Value": c["verified_source_versions"]},
    {"Metric": "Version coverage (%)", "Value": c["source_coverage_pct"]},
    {"Metric": "Raw reference papers verified", "Value": c["raw_reference_papers_verified"]},
]))
print("Trạng thái dữ liệu:", m1_summary["integrity_status"])
print("Kích thước source:", m1_summary["source_file_statistics"]
      or m1_summary["source_statistics_unavailable_reason"])
print("Benchmark pipeline mới:",
      m1_summary["benchmark_evidence"]["official_version_pipeline_metrics"])
print("Các bằng chứng còn thiếu:")
for item in m1_summary["missing_evidence"]:
    print(" -", item)
if EXPORT_M1_REPORT:
    print("Báo cáo tóm tắt:", export_m1_report(m1_summary))
else:
    print("✅ Chỉ xem thống kê, chưa xuất file mới.")

# Giữ cách trình bày 4 bảng như notebook nhánh main.
# Bảng legacy chỉ để xem/đối chiếu; không được coi là số liệu vN mới.
print("\n" + "=" * 65)
print("4 BẢNG THỐNG KÊ MILESTONE 1 (ĐỊNH DẠNG NOTEBOOK GỐC)")
print("=" * 65)
legacy_tables = [
    ("Bảng 1 — Khám phá metadata (cũ)", "m1_discovery_stats.csv"),
    ("Bảng 2 — Tải source cấp paper (legacy)", "m1_download_stats.csv"),
    ("Bảng 3 — Trích dẫn cấp paper (legacy)", "m1_reference_stats.csv"),
    ("Bảng 4 — Benchmark pipeline cũ (không phải benchmark vN)", "m1_system_benchmark.csv"),
]
for label, filename in legacy_tables:
    path = config.DEFAULT_DATA_DIR / "csv" / filename
    print(f"\n--- {label} ---")
    if not path.is_file():
        print("Chưa có bảng:", path)
        continue
    try:
        table = pd.read_csv(path, dtype={"arxiv_id": str})
        print("Tổng dòng:", len(table))
        display(table.head(10))
    except Exception as exc:
        print("Không thể đọc bảng:", filename, "|", exc)

version_path = config.DEFAULT_DATA_DIR / "csv" / "m1_download_versions.csv"
print("\n--- Bảng 2 bổ sung: tải source theo từng version ---")
if version_path.is_file():
    try:
        display(pd.read_csv(version_path, dtype={"arxiv_id": str}).head(10))
        if not integrity_report["snapshots"].get("download_versions_csv_current", False):
            print("⚠️ Bảng version đã cũ. Hãy xuất lại từ cell 6.")
    except Exception as exc:
        print("Không thể đọc version CSV:", exc)
else:
    print("Chưa xuất version CSV. Bật EXPORT_VERSION_CSV rồi chạy lại cell 6.")
